### Dataset Overview — Final Feature Sources (M1–M3)

This overview follows the structure of `Dataset_Overview2.ipynb`, but lists the **final selected feature sources** used to build `weekly_features.csv`, organised by model block (M1 market/macro-financial, M2 remote sensing, M3 shipping), as recorded in `external_sources.md`.

- Study period: **2006-01 – 2025-12**, aligned to **Friday-ending weekly frequency (W-FRI)**.
- All sources are public and free; only Global Fishing Watch (M3) requires a free API token.
- Numbering and variable names match `01_literature/beatrice_task_literature_matrix.md` (§①/§②/§③) and the build scripts under `03_data/processed/` and `04_code/scripts/`.

**M1) Core Market, Macro & Financial Variables**

10 recommended variables aligned with literature matrix §①, organised by Kilian's three mechanisms (supply / global demand / precautionary demand) plus market & financial conditions. All aligned to W-FRI, 2006–2025.

- **Existing** (#1–2, `vix` in #6): `03_data/processed/build_weekly_time_index.py` → `weekly_time_index.csv`
- **To build** (#3–5, `ovx` in #6, #7–10): `03_data/processed/build_m1_to_build.py` → `m1_to_build_weekly.csv`, then merged via `merge_m1_to_build.py`

| Dataset | Summary | URL | Variables | Time Range | Frequency | Coverage | Data Type | Access | Potential Use | Limitations |
|---------|---------|-----|-----------|------------|-----------|----------|-----------|--------|---------------|-------------|
| EIA Europe Brent Spot FOB | Official EIA Europe Brent spot price (FOB); own-price dynamics; local `EIA_brent_spot_price_daily*.xls` | [EIA](https://www.eia.gov/petroleum/gasdiesel/) | `brent_price` lags, `brent_log_return` = log(P_t / P_{t-1}) | 2006-01–present | Daily → weekly last (W-FRI) | Europe Brent; global crude benchmark | Price data | Open (local file) | Own oil-price momentum / lag features; target construction | Single price series; lags expanded at modelling stage (existing) |
| EIA Commercial Crude Stocks excl. SPR (WPSR) | Weekly US commercial crude inventories; supply / market balance; local `EIA_commercial_crude_stocks_weekly*.xls` | [EIA WPSR](https://www.eia.gov/petroleum/supply/weekly/) | `crude_stocks_change` (Δ inventory) | 2006-01–present | Weekly → W-FRI align, first difference | United States | Market / inventory data | Open (local file) | Inventory build/draw pressure | US-only; weekly may lag/smooth daily moves (existing) |
| Index of Global Real Economic Activity (Kilian) | Global demand proxy; Dallas Fed `igrea` (backup OECD CLI FRED `OECDLOLITOAASTSAM`) | [Dallas Fed](https://www.dallasfed.org/research/igrea) | `global_econ_activity` | 2006-02–present | Monthly → month-end ffill + 5-week release lag | Global | Macro activity index | Open | Global demand proxy (Kilian mechanism) | Monthly; conservative publication lag to avoid look-ahead (to build) |
| IMF Global Price Index of Industrial Materials | Non-oil industrial commodity prices; global demand; via FRED | [FRED PINDUINDEXM](https://fred.stlouisfed.org/series/PINDUINDEXM) | `nonoil_industrial_commodity` | 2006-02–present | Monthly → month-end ffill + 5-week lag | Global | Commodity price index | Open | Non-oil industrial demand proxy | Monthly; publication lag (to build) |
| Brent front-month futures − Brent spot | Market tightness / expectations; ICE `BZ=F` (Yahoo) − EIA spot (backup FRED `DCOILBRENTEU`) | [Yahoo BZ=F](https://finance.yahoo.com/quote/BZ%3DF) | `futures_spread` = log(fut) − log(spot) | 2007-08–present | Daily → weekly last | Brent | Futures / term structure | Open | Approximate term-structure / backwardation signal | Late start (2007-08); approximate term spread (to build) |
| CBOE OVX (priority) / CBOE VIX | Oil-specific & market uncertainty; Yahoo / FRED | [FRED VIXCLS](https://fred.stlouisfed.org/series/VIXCLS) | `ovx` (`^OVX`/FRED `OVXCLS`), `vix` (FRED `VIXCLS`) | OVX 2007-05; VIX 2006-01 | Daily → weekly last | US options market; global proxy | Volatility index | Open | Oil & market uncertainty; collinearity check | OVX late start; VIX not oil-specific (`ovx` to build, `vix` existing) |
| Geopolitical Risk Index (GPR) | Precautionary demand; Caldara & Iacoviello (2022); file `data_gpr_export.xls` col `GPR` | [GPR](https://www.matteoiacoviello.com/gpr.htm) | `gpr` | 2006-01–present | Monthly → ffill + 1-week release lag | Global | News-text aggregate index | Open | Low-frequency geopolitical-risk proxy | News-text aggregate; text modality removed (to build) |
| US 10-Year Treasury Yield change (ΔDGS10) | Rates / holding cost; first difference of FRED `DGS10` (local `treasury_10y`) | [FRED DGS10](https://fred.stlouisfed.org/series/DGS10) | `dgs10_change` (ΔDGS10) | 2006-01–present | Daily → weekly last → first difference | US bond market; global proxy | Interest rate data | Open | Rate-change / holding-cost control | Level failed unit-root test (P076) → use first difference (to build; source exists) |
| LBMA Gold Price PM (USD) | Commodity linkage / safe haven; via FRED (backup yfinance `GC=F`) | [FRED GOLDPMGBD228NLBM](https://fred.stlouisfed.org/series/GOLDPMGBD228NLBM) | `gold_return` (log return of `gold_price`) | 2006-01–present | Daily → weekly last → log return | Global | Commodity price | Open | Cross-commodity / risk linkage | Derived return series (to build) |
| Commodity-currency strength (CAD/USD, AUD/USD) | FX channel; mean of CAD/USD & AUD/USD; Yahoo (backup FRED `DEXCAUS`, `DEXUSAL`) | [Yahoo CADUSD=X](https://finance.yahoo.com/quote/CADUSD=X) | `commodity_fx` (mean weekly % change) | 2006-01–present | Daily → weekly last → mean % change | Canada/Australia FX; commodity proxy | Exchange rate data | Open | Commodity-exporter FX channel (vs broad USD) | Two-currency proxy; broad DXY evidence limited (to build) |

**Notes (echoing literature matrix §① re-reading):**
- Numbering maps one-to-one to §①; `brent_price` lags are expanded at the modelling stage, not listed as a separate variable (P076).
- #6 literature prefers **OVX over VIX** (P052); both kept for a collinearity check at modelling time.
- #8 level `treasury_10y` (DGS10) failed the unit-root test (P076); M1 uses the first difference `dgs10_change`.
- #10 broad USD index (DXY) evidence is limited (P053/P004); replaced by commodity-exporter FX CAD/AUD.
- `ovx`, `futures_spread` start late (OVX 2007-05; Brent futures 2007-08); early-week missing values are expected.
- `gpr` is a news-text aggregate index; the text modality was removed (Meeting 02) and `gpr` kept in M1 as a low-frequency geopolitical-risk proxy.
- Monthly variables (#3, #4, #7) use conservative publication lags to avoid look-ahead bias.
- Citations: GPR — Caldara & Iacoviello (2022) *AER*; Kilian REA — Kilian (2009), Dallas Fed updated edition.

**M2) Remote Sensing Variables**

Simplified three-layer design (literature matrix §②): ① dynamic NTL activity signal (within-site z-score anomaly, **not raw radiance**); ② remote-sensing observation quality; ③ Sentinel-2 daytime optical information availability + static capacity weights. Covers **11 oil-infrastructure AOIs** (centre + 5 km buffer; see `aoi_oil_infrastructure.csv`). VIIRS from 2014, S2 from 2017; all obtained free via Google Earth Engine.

- Build: GEE export → `build_m2_clean_features.py` → `weekly_m2_clean_features.csv` (44 cols = 4 types × 11 AOI) + `aoi_capacity_weights.csv`; merged via `04_code/scripts/build_feature_matrix.py`, registered in `feature_groups.json["M2_rs_clean"]`.

| Dataset | Summary | URL | Variables | Time Range | Frequency | Coverage | Data Type | Access | Potential Use | Limitations |
|---------|---------|-----|-----------|------------|-----------|----------|-----------|--------|---------------|-------------|
| NASA/NOAA VIIRS DNB Monthly (`avg_rad`, `cf_cvg`) | Nighttime-light radiance; used as **dynamic within-site z-score anomaly** (not raw level) + cloud-free coverage as data-quality | [GEE VIIRS](https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG) | `ntl_anomaly_{aoi}` (past-only expanding z-score, min 12 mo), `ntl_valid_obs_count_{aoi}` (=`cf_cvg`) | 2014-01–present | Monthly → daily ffill → W-FRI last | 11 oil AOIs × 5 km | Remote sensing (night lights) | Open (GEE) | Anchorage/port activity anomaly signal; observation quality | NTL↔tanker weak (Santos Rs=−0.07), **not a tanker proxy**; affected by AOI scale & urban light spill (P024/P032) |
| Copernicus Sentinel-2 SR + cloud prob (`valid_obs_count`, `cloud_probability`) | Daytime surface optical + cloud probability; daytime optical availability + information-gap proxy | [GEE S2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) | `s2_clear_obs_count_{aoi}`, `s2_cloud_fraction_{aoi}` (=`cloud_probability`/100) | 2017-04–present | Monthly → daily ffill → W-FRI last | 11 AOIs × 5 km | Remote sensing (optical) | Open (GEE) | Information-availability → returns mechanism (P025) | Cloud cover reduces usable obs; 10 m too coarse for vessel/tank detection |
| AOI capacity weights (static) | Static scale proxy derived from long-term VIIRS mean radiance; cross-AOI capacity weighting | — `aoi_capacity_weights.csv` | `aoi_capacity_weight` | static | static | 11 AOIs (normalised) | Derived static weight | Open (derived) | Capacity-weighted cross-AOI aggregation (**not a high-frequency feature**) | Static snapshot; cross-section NTL↔port-size Rs=0.69–0.84 (P024) |

**Notes (echoing literature matrix §② re-reading):**
- ⚠️ **Dynamic anomaly, not raw level**: renamed from old `ntl_ntl_avg_rad_mean_{P00x}` to `ntl_anomaly_{aoi}` (within-site z-score), since NTL is temporally weak and raw level is driven by AOI scale / urban light pollution (P024/P032).
- ⚠️ **NTL is not a tanker proxy**: Santos NTL↔tanker Rs=−0.07 → use only as a composite anchorage/port activity signal; combine with tanker share / AIS at modelling time (P024).
- ⚠️ **VIIRS (not DMSP) + post-2012 subsample** (VCMSLCFG from 2014); GEE-side masking applied; missing values only filled with history to avoid look-ahead leakage.
- ⚠️ **`valid_obs` role**: VIIRS `cf_cvg` is a **data-quality** variable first; the P025 information-availability → returns mechanism maps to S2 daytime cloud-free obs (`s2_cloud_fraction`).
- ⚠️ **No floating-roof fill level**: P055 estimates tank structural volume (V=πr²h) requiring sub-metre imagery (not reproducible with S2 10 m); replaced by long-term mean radiance as `aoi_capacity_weight`.
- ⚠️ **Sample alignment**: with `s2_*`, M2 weekly sample starts 2017; remote sensing is a demand-side/upstream information proxy; incremental value proven via "M1 vs M1+M2" ablation + Clark–West/DM tests (P069), not by directly predicting price.
- Citations: NTL shipping proxy — Polinov, Bookman & Levin (2022); NTL data choice — Gibson et al. (2021); cloud & oil-price info — Hao & Wang (2023); satellite oil-demand — Bricongne et al. (2026, ECB WP 3198); tank capacity — Wang et al. (2019).

**M3) Shipping & Port Activity Variables**

Tanker-specific (literature matrix §③): flow intensity + capacity (DWT) weighting + average vessel size + region/chokepoint split + export–import directional asymmetry + congestion proxy. Aligned to W-FRI, covering **6 oil-related chokepoints** (Hormuz / Suez / Malacca / Bab el-Mandeb / Panama / Cape). PortWatch is free and key-less; GFW needs a free API token.

- Build: raw download (`03_data/raw/05_shipping/`) → `aggregate_shipping_to_weekly.py` → `weekly_shipping_features.csv` → merged into `weekly_features.csv`, registered in `feature_groups.json["M3_add_shipping"]` (119 candidate features). Helper scripts: `add_avg_tanker_size.py`, `sync_shipping_features.py`.

| Dataset | Summary | URL | Variables | Time Range | Frequency | Coverage | Data Type | Access | Potential Use | Limitations |
|---------|---------|-----|-----------|------------|-----------|----------|-----------|--------|---------------|-------------|
| IMF PortWatch `Daily_Chokepoints_Data` | Chokepoint transits by vessel type (`n_*`/`capacity_*`); coarse seaborne tanker flow | [IMF PortWatch](https://portwatch.imf.org/) | `pw_{choke}_n_tanker`, `_capacity_tanker`, `_tanker_share`, `_tanker_cap_share`, `_avg_tanker_size`, `_n_tanker_wow_pct`, `_capacity_tanker_4w_ma`; aggregates `pw_all_n_tanker_sum`/`_n_total_sum`/`_tanker_share` | 2019-01–present | Daily → W-FRI sum | 6 chokepoints (Hormuz/Suez/Malacca/Bab el-Mandeb/Panama/Cape) | Shipping / chokepoint transit | Open (ArcGIS FeatureServer) | Tanker traffic-intensity proxy; disruption detection | Not exact oil trade volume; chokepoint data has **no direction field** (P016/P018) |
| IMF PortWatch `Daily_Ports_Data` | Port-level tanker import/export tonnage at 14 oil hubs; directional asymmetry | [IMF PortWatch](https://portwatch.imf.org/) | `pw_exp_hubs_export_vol`, `pw_imp_hubs_import_vol`, `pw_tanker_exp_imp_net`/`_asym`/`_log_ratio`/`_asym_4w_ma`, `pw_exp_hubs_export_vol_wow_pct` | 2019-01–present | Daily → W-FRI sum (export/import baskets) | 14 tanker hubs (export/import baskets) | Shipping / trade flow | Open (ArcGIS FeatureServer) | Directional export–import asymmetry | Tonnage estimated from AIS draft (no embedded price → leak-safe); chokepoint data lacks direction |
| Global Fishing Watch 4Wings `public-global-presence` | AIS vessel-presence hours/vessels by type; congestion/dwell proxy | [GFW APIs](https://globalfishingwatch.org/our-apis/) | `gfw_{choke}_total_hours`/`_total_vessels`/`_cargo_hours`/`_bunker_hours`/`_other_hours`/`_nontanker_hours`/`_other_share`/`_total_hours_mom_pct`/`_dwell_hours_per_vessel`; `gfw_all_total_hours_sum` | 2012-01–present | Monthly → daily ffill → W-FRI last | 6 chokepoint polygons | Shipping / AIS presence | Open (Report API, Bearer token) | Congestion / dwell-time proxy (hours/vessels) | Monthly; no dedicated anchorage-waiting data; tanker filtering approximate |

**Export / import hub baskets (port-level directionality, `download_portwatch_ports.py`):**
- **Export hubs**: Ras Tanura, Juaymah, Yanbu (Saudi), Ras Laffan (Qatar), Primorsk, Novorossiysk (Russia), Corpus Christi (US), Sidi Kerir (Egypt/SUMED), Bonny (Nigeria).
- **Import / refining hubs**: Rotterdam (NL), Singapore, Ningbo (China), Chiba (Japan), Ulsan (Korea).
- Roles validated against data (export hubs `export_tanker ≫ import_tanker`, import hubs the reverse).

**Notes (echoing literature matrix §③ re-reading):**
- ⚠️ `pw_{choke}_n_tanker` is a **coarse tanker traffic / seaborne flow proxy**, not P016 port-call frequency nor exact oil trade volume (P016/P018).
- ⚠️ **Directionality** uses PortWatch **port-level** `import_tanker`/`export_tanker` (AIS-draft tonnage, no embedded price → leak-safe); PortWatch **chokepoint data has no direction field**, so directionality cannot be built from chokepoint data (P018).
- ⚠️ **Congestion**: no dedicated anchorage-waiting data; use GFW presence `total_hours/total_vessels` (dwell per vessel) as a dwell proxy (P016 dwell-time concept).
- ⚠️ **Sample alignment**: PortWatch from 2019+, GFW from 2012+; oil-price direction is price→shipping (P016/P017), so modelling must use **strict lags, release-time alignment and backward rolling** (avoid P018 centred-MA look-ahead leakage).
- Citations: PortWatch — Arslanalp et al. (2026, IMF WP/26/99); Arslanalp, Marini & Tumbarello (2019, IMF WP/19/275). GFW — Global Fishing Watch 4Wings API. Oil–shipping — Mi et al. (2022, 2023).